In [ ]:
!pip install pandas requests plotly -q

import requests
import time
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# 1. GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT'],
    'INTERVAL': '1h',          # BẮT BUỘC LÀ 1H cho chiến lược này
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# 2. DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)
        end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
        end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {self.symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": self.symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        return df.astype(float)

# ==========================================
# 3. STRATEGY LOGIC (RETURNS)
# ==========================================
class FlashCrashStrategy:
    def __init__(self):
        # Thông số tỷ lệ %
        self.trend_pct = 0.03   # Tăng 3%
        self.crash_pct = -0.01  # Giảm 1%
        self.tp_pct = 0.04      # Lời 4%
        self.sl_pct = 0.03      # Lỗ 3%

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # Ret_1h: Tính % thay đổi của 1 nến 1H
        df['Ret_1h'] = df['Close'].pct_change(periods=1)
        # Ret_6h: Tính % thay đổi của 6 nến 1H
        df['Ret_6h'] = df['Close'].pct_change(periods=6)

        df.dropna(inplace=True)
        return df

# ==========================================
# 4. BACKTEST ENGINE
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades = []
        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values

        ret_1h = self.df['Ret_1h'].values
        ret_6h = self.df['Ret_6h'].values
        times = self.df.index

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']
        s = self.strategy

        self.equity_curve[0] = self.capital

        for i in range(1, len(self.df)):

            # --- EXIT LOGIC (FIXED TP/SL) ---
            if self.position == 1:
                exit_signal = False
                reason = ""

                tp_price = self.entry_price * (1 + s.tp_pct)
                sl_price = self.entry_price * (1 - s.sl_pct)

                # Kiểm tra chạm SL trước (ưu tiên an toàn)
                if L[i] <= sl_price:
                    exit_signal = True
                    reason = "Stop Loss (-3%)"
                    raw_exit_price = min(O[i], sl_price) # Xử lý Gap mở cửa

                # Kiểm tra chạm TP
                elif H[i] >= tp_price:
                    exit_signal = True
                    reason = "Take Profit (+4%)"
                    raw_exit_price = max(O[i], tp_price) # Xử lý Gap mở cửa

                if exit_signal:
                    real_exit = raw_exit_price * (1 - slip)
                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long', 'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })
                    self.position = 0

            # --- ENTRY LOGIC ---
            if self.position == 0:
                # Điều kiện: Tăng 3% trong 6H VÀ Giảm 1% trong 1H
                cond_trend = ret_6h[i-1] >= s.trend_pct
                cond_crash = ret_1h[i-1] <= s.crash_pct

                if cond_trend and cond_crash:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

            # CẬP NHẬT EQUITY
            current_equity = self.capital
            if self.position == 1:
                unrealized_pnl = self.invested_amount * ((C[i] - self.entry_price) / self.entry_price)
                current_equity += unrealized_pnl
            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# 5. EVALUATOR (BÁO CÁO & PLOT)
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(df_raw, equity, trades_df, config, symbol):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        initial_capital = config['INITIAL_CAPITAL']
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 else float('inf')

        print(f"\n{'='*50}")
        print(f"📊 REPORT: {symbol} | Strat: Uptrend Flash Crash")
        print(f"{'='*50}")
        print(f"Final Equity:   ${final_equity:,.2f}")
        print(f"Net PnL:        ${net_pnl:,.2f}")
        print(f"Total Return:   {total_return:.2%}")
        print(f"Max Drawdown:   {max_dd:.2%}")
        print(f"Total Trades:   {len(trades_df)}")
        print(f"Winrate:        {winrate:.2%}")
        print(f"Profit Factor:  {profit_factor:.2f}")
        print(f"{'='*50}\n")

        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                            row_heights=[0.5, 0.3, 0.2], subplot_titles=(f"Price ({symbol})", "Equity Curve", "Drawdown"))
        fig.add_trace(go.Candlestick(x=df_raw.index, open=df_raw['Open'], high=df_raw['High'], low=df_raw['Low'], close=df_raw['Close'], name="Price"), row=1, col=1)
        fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines", name="Equity", line=dict(color="#2962ff")), row=2, col=1)
        fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown.values, mode="lines", name="DD", line=dict(color="#d32f2f"), fill='tozeroy'), row=3, col=1)

        fig.update_layout(title=f"Strategy 25: {symbol}", height=800, xaxis_rangeslider_visible=False, showlegend=False)
        fig.show()

if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()
        if not df_raw.empty:
            engine = BacktestEngine(df_raw, FlashCrashStrategy(), CONFIG)
            equity, trades = engine.run()
            Evaluator.generate_report(df_raw, equity, trades, CONFIG, symbol)


📊 REPORT: BTCUSDT | Strat: Uptrend Flash Crash
Final Equity:   $10,117.43
Net PnL:        $117.43
Total Return:   1.17%
Max Drawdown:   -0.36%
Total Trades:   3
Winrate:        100.00%
Profit Factor:  inf




📊 REPORT: ETHUSDT | Strat: Uptrend Flash Crash
Final Equity:   $10,023.65
Net PnL:        $23.65
Total Return:   0.24%
Max Drawdown:   -0.53%
Total Trades:   6
Winrate:        50.00%
Profit Factor:  1.25

